# Banners and Dialogs

**Part I · Visualization** — Tutorial 18

Show transient overlays and dialogs over the viewer. You will learn to:

- Show acknowledge banners (`alert`), custom-option banners (`show_banner`),
  and yes/no/cancel dialogs (`confirm`).
- Use the modal variant (`show_banner(..., dismissable=False)`).
- Control alignment, global vs per-scene scope, and auto-hide vs explicit
  removal.
- Await the push from an async handler with `show_banner_async()`.

> **No graphical output in these docs.** These cells build the viewer state and
> flush it, but they don't open the viewer — so no image is rendered here. Run the
> notebook in Jupyter to see the result, or open a separate browser window with
> `viz.show(jupyter=False)`.


## Setup


In [1]:
from pytanga.geometry import Point
from pytanga.viz import Button, Slider, Visualizer


## 1. `alert()` — acknowledge

A single "OK" acknowledge banner (auto-hides on click).


In [2]:
viz = Visualizer(add_default_axes=False, add_default_grid=False)
# viz.show(jupyter=False)  # open a separate browser window to see the live viewer
viz.add(Point(0, 0, 0), color="#ff4444")

bid = viz.alert("## Done\n\nThis is an **acknowledge** banner.", title="Notice")
# viz.remove_banner(bid)   # explicit removal
viz.flush()
print("alert shown")


alert shown


## 2. `show_banner()` — custom options

`show_banner(text, controls=[Button(...), Slider(...), ...])` lays out custom
controls in the banner. Markdown/KaTeX text is supported.


In [3]:
viz = Visualizer(add_default_axes=False, add_default_grid=False)
viz.add(Point(0, 0, 0), color="#ff4444")

bid = viz.show_banner(
    "Choose an option:",
    title="Options",
    controls=[
        Button(id="opt_a", label="Option A"),
        Button(id="opt_b", label="Option B"),
        Slider(id="s", label="Level", min=0, max=10, value=5),
    ],
    align_x=0.2,
    align_y=0.2,
)
viz.flush()
print("options banner shown")


options banner shown


## 3. `confirm()` — yes / no / cancel

A convenience dialog with `on_yes` / `on_no` / `on_cancel` callbacks.


In [4]:
viz = Visualizer(add_default_axes=False, add_default_grid=False)
viz.add(Point(0, 0, 0), color="#ff4444")

async def on_yes(value, event):
    print("confirmed")

bid = viz.confirm("Proceed with the calculation?", title="Confirm", on_yes=on_yes)
viz.flush()
print("confirm dialog shown")


confirm dialog shown


## 4. Modal banner — `dismissable=False`

A modal banner draws a dimmed backdrop and blocks the scene until removed.


In [5]:
viz = Visualizer(add_default_axes=False, add_default_grid=False)
viz.add(Point(0, 0, 0), color="#ff4444")

bid = viz.show_banner(
    "## Busy…\n\nComputing…",
    title="Working",
    dismissable=False,
)
# viz.remove_banner(bid)   # unblock
viz.flush()
print("modal banner shown")


modal banner shown


## 5. Alignment, scope, and auto-hide

`align_x` / `align_y` (both in `[0, 1]`) pin the banner within its container.
`scene_name=None` (default) is **global**; `scene_name="<name>"` (or a
`VizSceneHandle`) is **per-scene**. `auto_hide=True` (default) removes the
banner when the user selects an option; otherwise remove it explicitly with
`remove_banner(id)` / `clear_banners()`.


In [6]:
viz = Visualizer(add_default_axes=False, add_default_grid=False)
viz.add(Point(0, 0, 0), color="#ff4444")

# Global, centered:
bid_global = viz.show_banner("Global")

# Per-scene (only over the detail scene):
detail = viz.scene("detail")
detail.show_banner("Only over the detail scene")

# Explicit removal (auto_hide=False):
bid = viz.show_banner("Please wait…", dismissable=False, auto_hide=False)
viz.remove_banner(bid)
# viz.clear_banners()
viz.flush()
print("scope / auto-hide demonstrated")


scope / auto-hide demonstrated


## 6. `show_banner_async()` — await the push

Inside an async handler, `await show_banner_async(...)` guarantees the banner
is pushed before continuing (see
[Tutorial 19](../19_responsive_computation/) for the full pattern).


In [7]:
# Inside an async handler:
#   bid = await viz.show_banner_async("## Calculating…", dismissable=False)
#   ... run work ...
#   viz.remove_banner(bid)

print("show_banner_async() awaits the banner push")


show_banner_async() awaits the banner push


## 7. Dialog overlay — `show_dialog()`

For richer, titled overlays (arbitrary `View` content, a menu bar, a file
picker), use the dialog API (1.15.0):

- `show_dialog(content, id=..., title=..., dismissable=...)` renders a titled
  dialog whose body holds arbitrary view content; `dismissable=False` makes it
  modal (dimmed backdrop, no ✕). It also accepts `width=` / `height=` (px or
  % of viewport) and a drag-to-resize corner.
- `remove_dialog(id)` / `clear_dialogs()` dismiss dialogs.
- `FileChooserDialog(control_id, ...)` is a ready-made file-open dialog with a
  path line and OK/Cancel (`on_accept` / `on_close`).


## Visual Examples

Every banner kind in one scene (the library's `banner_types.py` pattern).


In [8]:
viz = Visualizer(add_default_axes=False, add_default_grid=False)
viz.add(Point(0, 0, 0), color="#ff4444")

# Acknowledge:
bid1 = viz.alert("## Welcome\n\nAn **acknowledge** banner.", title="Notice")

# Custom options:
bid2 = viz.show_banner(
    "Choose an option:",
    title="Options",
    controls=[Button(id="a", label="Option A"), Button(id="b", label="Option B")],
    align_x=0.2, align_y=0.2,
)

# Yes / No / Cancel:
bid3 = viz.confirm("Proceed?", title="Confirm")

# Modal:
bid4 = viz.show_banner("## Busy…", title="Working", dismissable=False)

viz.flush()
print("all banner kinds shown")


all banner kinds shown


## Summary

| Banner | API |
|---|---|
| Acknowledge | `viz.alert(text, title=..., on_ok=...)` |
| Custom options | `viz.show_banner(text, controls=[Button(...), Slider(...)], align_x=, align_y=)` |
| Yes/no/cancel | `viz.confirm(text, on_yes=, on_no=, on_cancel=)` |
| Modal | `viz.show_banner(text, dismissable=False)` |
| Per-scene | `scene_name="<name>"` / `VizSceneHandle.show_banner(...)` |
| Removal | `remove_banner(id)` / `clear_banners()` |
| Await push | `await show_banner_async(...)` |

**Next:** [19 — Responsive Computation](../19_responsive_computation/).
